# 🚗 Tesla Deliveries — End-to-End ML Pipeline (2015–2025)
### Preprocessing · EDA · Feature Engineering · Regression · Time-Series Forecasting

> **Dataset** : `tesla_deliveries_dataset_2015_2025.csv`  
> **Goal** : Predict *Estimated Deliveries* and forecast future delivery trends  
> **Audience** : Beginner to intermediate

---
### 📌 Table of Contents
1. [Imports](#imports)  
2. [Data Loading & First Look](#data-loading)  
3. [Exploratory Data Analysis](#eda)  
4. [Preprocessing & Feature Engineering](#preprocessing)  
5. [Regression Modeling](#regression)  
6. [Hyperparameter Tuning](#tuning)  
7. [Time-Series Forecasting (SARIMA)](#sarima)  
8. [Summary & Key Insights](#summary)

---
> ⚠️ **Important note on model performance**  
> After removing data-leakage columns (`Production_Units`, `CO2_Saved_tons`),  
> the remaining features have near-zero correlation with the target.  
> This is a property of the **synthetic dataset** — not a modeling mistake.  
> Regression models are included for learning purposes; SARIMA is the valid forecasting approach.

## 1 · Imports <a id='imports'></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
from sklearn.linear_model    import LinearRegression, Ridge, Lasso
from sklearn.ensemble        import RandomForestRegressor
from sklearn.metrics         import (mean_absolute_error, mean_squared_error,
                                     r2_score, mean_absolute_percentage_error)

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools          import adfuller

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print("✅ Libraries loaded")

## 2 · Data Loading & First Look <a id='data-loading'></a>

We load the CSV and do a quick sanity check — shape, column types, missing values.

| Column | Description |
|--------|-------------|
| `Year`, `Month` | Time dimension |
| `Region` | Europe / Asia / North America / Middle East |
| `Model` | Tesla model (S, X, 3, Y, Cybertruck) |
| `Estimated_Deliveries` | 🎯 **Prediction target** |
| `Production_Units` | Units manufactured — **dropped (leakage)** |
| `CO2_Saved_tons` | Derived from deliveries — **dropped (leakage)** |
| `Avg_Price_USD` | Average sale price |
| `Battery_Capacity_kWh` | Battery size |
| `Range_km` | Driving range |
| `Charging_Stations` | Available chargers |
| `Source_Type` | Data reliability tag |

In [ ]:
df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')

print(f"Shape       : {df.shape}")
print(f"Date range  : {df['Year'].min()} – {df['Year'].max()}")
print(f"Missing     : {df.isnull().sum().sum()} cells")
print(f"\nRows per Year/Month: {df.groupby(['Year','Month']).size().unique()} (always 20 = 4 regions × 5 models)")

df.head()

In [ ]:
df.describe().round(2)

## 3 · Exploratory Data Analysis <a id='eda'></a>

We explore delivery trends, model and region share, feature correlations,  
target distribution, and outliers before touching the model.

In [ ]:
# ── 3-A  Yearly delivery trend ────────────────────────────────────────────────
yearly = df.groupby('Year')['Estimated_Deliveries'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly['Year'], yearly['Estimated_Deliveries'] / 1e6,
       color=sns.color_palette('muted')[0], edgecolor='white', width=0.6)
ax.set_title('Total Estimated Deliveries by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Deliveries (Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))

for row in yearly.itertuples():
    ax.text(row.Year, row.Estimated_Deliveries / 1e6 + 0.01,
            f"{row.Estimated_Deliveries/1e6:.2f}M", ha='center', fontsize=8)

plt.tight_layout()
plt.show()
print("📌 Deliveries are relatively stable across years — a characteristic of this synthetic dataset.")

In [ ]:
# ── 3-B  Deliveries by Model ──────────────────────────────────────────────────
by_model = df.groupby('Model')['Estimated_Deliveries'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

by_model.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'), edgecolor='white', rot=0)
axes[0].set_title('Total Deliveries by Model', fontweight='bold')
axes[0].set_ylabel('Deliveries')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

axes[1].pie(by_model, labels=by_model.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2'), startangle=140)
axes[1].set_title('Delivery Share by Model', fontweight='bold')

plt.tight_layout()
plt.show()
print("📌 All models have nearly equal share — confirming the synthetic/uniform distribution.")

In [ ]:
# ── 3-C  Deliveries by Region ─────────────────────────────────────────────────
by_region = df.groupby('Region')['Estimated_Deliveries'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(by_region.index, by_region.values / 1e6,
               color=sns.color_palette('muted', 4), edgecolor='white')
ax.set_title('Total Deliveries by Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Deliveries (Millions)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))

for bar, val in zip(bars, by_region.values):
    ax.text(val / 1e6 + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.2f}M', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 3-D  Correlation Heatmap ──────────────────────────────────────────────────
#
# KEY FINDING:
# Production_Units (0.994) and CO2_Saved_tons (0.837) are near-perfect
# predictors BUT they are data leakage — derived from deliveries.
# All other features have correlation ≤ 0.03 with the target.

numeric_cols = ['Estimated_Deliveries', 'Production_Units', 'CO2_Saved_tons',
                'Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km',
                'Charging_Stations', 'Year', 'Month']

corr = df[numeric_cols].corr().round(2)

# ── Figure layout: heatmap on left, bar chart on right ──
fig, axes = plt.subplots(1, 2, figsize=(18, 7),
                         gridspec_kw={'width_ratios': [2, 1]})

# --- Left: full heatmap ---
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # hide upper triangle

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    ax=axes[0],
    linewidths=0.8,
    linecolor='white',
    square=True,
    annot_kws={'size': 9},
    cbar_kws={'shrink': 0.8, 'label': 'Correlation coefficient'}
)

axes[0].set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=40, ha='right', fontsize=9)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=9)

# Highlight leaky columns with a red border box
for spine in axes[0].spines.values():
    spine.set_visible(True)

# --- Right: correlation with target (bar chart) ---
target_corr = corr['Estimated_Deliveries'].drop('Estimated_Deliveries').sort_values()

colors = ['#d73027' if abs(v) > 0.5 else '#4575b4' if v > 0 else '#74add1'
          for v in target_corr]

bars = axes[1].barh(target_corr.index, target_corr.values,
                    color=colors, edgecolor='white', height=0.6)

axes[1].axvline(x=0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].axvline(x=0.5,  color='red', linewidth=1, linestyle=':', alpha=0.6, label='±0.5 threshold')
axes[1].axvline(x=-0.5, color='red', linewidth=1, linestyle=':', alpha=0.6)

for bar, val in zip(bars, target_corr.values):
    axes[1].text(val + (0.01 if val >= 0 else -0.01),
                 bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center',
                 ha='left' if val >= 0 else 'right', fontsize=9)

axes[1].set_title('Correlation with Target\n(Estimated_Deliveries)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pearson Correlation')
axes[1].set_xlim(-0.2, 1.1)
axes[1].legend(fontsize=8)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout(pad=2)
plt.show()

print("🔴 RED bars = data leakage (Production_Units, CO2_Saved_tons) — must be dropped")
print("🔵 BLUE bars = safe features — but correlation ≤ 0.03, very weak signal")
print("\n📌 After removing leakage, no feature has meaningful predictive power.")
print("   This is a property of the synthetic dataset, not a modeling error.")

In [ ]:
# ── 3-E  Target Variable Distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['Estimated_Deliveries'], bins=50,
             color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_title('Distribution of Estimated Deliveries', fontweight='bold')
axes[0].set_xlabel('Deliveries')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['Estimated_Deliveries']), bins=50,
             color=sns.color_palette('muted')[1], edgecolor='white')
axes[1].set_title('Log-Transformed Deliveries', fontweight='bold')
axes[1].set_xlabel('log(1 + Deliveries)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()
print("📌 Right-skewed distribution. Log transform makes it more symmetric.")

In [ ]:
# ── 3-F  Average Price by Model Over Years ────────────────────────────────────
price_trend = df.groupby(['Year', 'Model'])['Avg_Price_USD'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for model, grp in price_trend.groupby('Model'):
    ax.plot(grp['Year'], grp['Avg_Price_USD'], marker='o', label=model, linewidth=2)

ax.set_title('Average Price by Model Over Years', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Price (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 3-G  Outlier Detection ────────────────────────────────────────────────────
outlier_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
                'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=df[outlier_cols], orient='h', palette='Set3', ax=ax)
ax.set_title('Outlier Detection Across Key Numerical Features', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature Values')
plt.tight_layout()
plt.show()
print("📌 A small number of outliers are present. Tree-based models (Random Forest) are robust to these.")

## 4 · Preprocessing & Feature Engineering <a id='preprocessing'></a>

**Steps:**
1. Sort chronologically  
2. Drop leaky columns (`Production_Units`, `CO2_Saved_tons`)  
3. Add cyclical month features (`month_sin`, `month_cos`)  
4. Encode categorical columns with `OneHotEncoder`  
5. Scale numerical columns with `StandardScaler`  
6. Chronological 80/20 train-test split  

> **Why cyclical encoding for months?**  
> Month 12 and Month 1 are neighbours, but numerically far apart.  
> `sin(2π × month/12)` and `cos(2π × month/12)` preserve that circular relationship.

In [ ]:
# Sort chronologically
df = df.sort_values(['Year', 'Month']).reset_index(drop=True)

# ── Feature engineering ───────────────────────────────────────────────────────
df['month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# ── Feature selection ─────────────────────────────────────────────────────────
# NOTE: Production_Units (corr=0.994) and CO2_Saved_tons (corr=0.837) are
# excluded — they are derived from / computed alongside Estimated_Deliveries.
# Using them would be data leakage (the model would be "cheating").
# After removing these, remaining features have correlation ≤ 0.03 with target.

categorical_cols = ['Region', 'Model', 'Source_Type']
numerical_cols   = ['Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km',
                    'Charging_Stations', 'month_sin', 'month_cos']

X = df[categorical_cols + numerical_cols]
y = df['Estimated_Deliveries']

# ── Chronological train/test split ────────────────────────────────────────────
split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Train : {len(X_train):,} rows  ({df.iloc[:split_index]['Year'].min()}–{df.iloc[:split_index]['Year'].max()})")
print(f"Test  : {len(X_test):,} rows  ({df.iloc[split_index:]['Year'].min()}–{df.iloc[split_index:]['Year'].max()})")

# ── Preprocessing pipeline ────────────────────────────────────────────────────
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                        numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'),  categorical_cols)
])

print("\n✅ Preprocessing pipeline created")

## 5 · Regression Modeling <a id='regression'></a>

We train five models and compare them on MAE, RMSE, and R².

| Model | Description |
|-------|-------------|
| Baseline | Always predicts the training mean |
| Linear Regression | Best-fit linear surface |
| Ridge | Linear + L2 penalty (prevents large coefficients) |
| Lasso | Linear + L1 penalty (drives weak features to zero) |
| Random Forest | 100 decision trees voting together |

> ⚠️ **Expected result:** All models will have negative or near-zero R² because  
> the non-leaky features have near-zero correlation with the target in this synthetic dataset.  
> This is a valid and important finding — no algorithm can learn from uncorrelated features.

In [ ]:
# ── Baseline model ────────────────────────────────────────────────────────────
baseline_pred = np.full(len(y_test), y_train.mean())

baseline_mae  = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mape = mean_absolute_percentage_error(y_test, baseline_pred)

print(f"Baseline (predict mean = {y_train.mean():,.0f})")
print(f"  MAE  : {baseline_mae:,.2f}")
print(f"  RMSE : {baseline_rmse:,.2f}")
print(f"  MAPE : {baseline_mape:.4f}")

In [ ]:
# ── Train all regression models ───────────────────────────────────────────────

models = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(alpha=1.0, random_state=42),
    'Lasso'             : Lasso(alpha=0.1, max_iter=10000),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

results = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results[name] = {
        'pipeline' : pipe,
        'pred'     : pred,
        'MAE'      : mean_absolute_error(y_test, pred),
        'RMSE'     : np.sqrt(mean_squared_error(y_test, pred)),
        'R2'       : r2_score(y_test, pred),
        'MAPE'     : mean_absolute_percentage_error(y_test, pred),
    }
    print(f"{name:<22} | MAE={results[name]['MAE']:>8,.0f} | RMSE={results[name]['RMSE']:>8,.0f} | R²={results[name]['R2']:>7.4f}")

# Keep references for later cells
rf_pipeline = results['Random Forest']['pipeline']
rf_pred     = results['Random Forest']['pred']
lr_mae      = results['Linear Regression']['MAE']
lr_rmse     = results['Linear Regression']['RMSE']
lr_r2       = results['Linear Regression']['R2']
lr_mape     = results['Linear Regression']['MAPE']
lr_pred     = results['Linear Regression']['pred']

ridge_mae   = results['Ridge']['MAE']
ridge_rmse  = results['Ridge']['RMSE']
ridge_r2    = results['Ridge']['R2']
ridge_mape  = results['Ridge']['MAPE']
ridge_pred  = results['Ridge']['pred']

lasso_mae   = results['Lasso']['MAE']
lasso_rmse  = results['Lasso']['RMSE']
lasso_r2    = results['Lasso']['R2']
lasso_mape  = results['Lasso']['MAPE']
lasso_pred  = results['Lasso']['pred']

rf_mae      = results['Random Forest']['MAE']
rf_rmse     = results['Random Forest']['RMSE']
rf_r2       = results['Random Forest']['R2']
rf_mape     = results['Random Forest']['MAPE']

In [ ]:
# ── Model comparison ──────────────────────────────────────────────────────────

comparison_df = pd.DataFrame([
    {'Model': 'Baseline',          'MAE': baseline_mae,  'RMSE': baseline_rmse, 'R2': np.nan,  'MAPE': baseline_mape},
    {'Model': 'Linear Regression', 'MAE': lr_mae,         'RMSE': lr_rmse,       'R2': lr_r2,   'MAPE': lr_mape},
    {'Model': 'Ridge',             'MAE': ridge_mae,      'RMSE': ridge_rmse,    'R2': ridge_r2,'MAPE': ridge_mape},
    {'Model': 'Lasso',             'MAE': lasso_mae,      'RMSE': lasso_rmse,    'R2': lasso_r2,'MAPE': lasso_mape},
    {'Model': 'Random Forest',     'MAE': rf_mae,         'RMSE': rf_rmse,       'R2': rf_r2,   'MAPE': rf_mape},
]).sort_values('MAE').reset_index(drop=True)

display(comparison_df.style.format({'MAE':'{:,.0f}','RMSE':'{:,.0f}','R2':'{:.4f}','MAPE':'{:.4f}'}))

# Bar chart of MAE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c' if r2 < 0 else '#2ecc71' for r2 in comparison_df['R2'].fillna(0)]

axes[0].barh(comparison_df['Model'], comparison_df['MAE'], color=colors, edgecolor='white')
axes[0].set_title('MAE by Model (lower = better)', fontweight='bold')
axes[0].set_xlabel('Mean Absolute Error')
axes[0].invert_yaxis()

axes[1].barh(comparison_df['Model'].iloc[1:], comparison_df['R2'].iloc[1:], color=colors[1:], edgecolor='white')
axes[1].axvline(x=0, color='black', linewidth=1, linestyle='--')
axes[1].set_title('R² Score (higher = better, 0 = baseline)', fontweight='bold')
axes[1].set_xlabel('R² Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n📌 All models score near or below zero R² — expected given near-zero feature correlations.")
print("   The synthetic dataset has no learnable signal in the non-leaky features.")

In [ ]:
# ── Feature importance (Random Forest) ───────────────────────────────────────

encoded_cat = list(
    rf_pipeline.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(categorical_cols)
)
all_features = numerical_cols + encoded_cat

fi_df = pd.DataFrame({
    'Feature'    : all_features,
    'Importance' : rf_pipeline.named_steps['model'].feature_importances_
}).sort_values('Importance', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='Blues_r', ax=ax)
ax.set_title('Top 12 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
print("📌 Even the top features score low — confirming that no feature drives deliveries in this dataset.")

In [ ]:
# ── Actual vs Predicted & Residual Analysis ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test, rf_pred, alpha=0.5, color='steelblue')
lims = [min(y_test.min(), rf_pred.min()), max(y_test.max(), rf_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_title('Actual vs Predicted (Random Forest)', fontweight='bold')
axes[0].set_xlabel('Actual Deliveries')
axes[0].set_ylabel('Predicted Deliveries')
axes[0].legend()

# 2. Residual distribution
residuals = y_test - rf_pred
sns.histplot(residuals, bins=30, kde=True, ax=axes[1], color='coral')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (Actual − Predicted)')

# 3. Residuals vs Predicted
axes[2].scatter(rf_pred, residuals, alpha=0.5, color='steelblue')
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[2].set_title('Residuals vs Predicted Values', fontweight='bold')
axes[2].set_xlabel('Predicted Deliveries')
axes[2].set_ylabel('Residual')

plt.tight_layout()
plt.show()

print(f"Mean Residual   : {residuals.mean():,.0f}")
print(f"Median Residual : {residuals.median():,.0f}")
print(f"Std Residual    : {residuals.std():,.0f}")

## 6 · Hyperparameter Tuning <a id='tuning'></a>

We tune Ridge and Lasso's `alpha` parameter using `GridSearchCV` with `TimeSeriesSplit`  
to preserve temporal order during cross-validation.

In [ ]:
# ── Ridge tuning ──────────────────────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5)

ridge_search = GridSearchCV(
    estimator  = Pipeline([('preprocessor', preprocessor), ('model', Ridge())]),
    param_grid = {'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100]},
    cv         = tscv,
    scoring    = 'neg_mean_absolute_error',
    n_jobs     = -1
)
ridge_search.fit(X_train, y_train)
ridge_best_pred = ridge_search.best_estimator_.predict(X_test)
ridge_best_mae  = mean_absolute_error(y_test, ridge_best_pred)
ridge_best_r2   = r2_score(y_test, ridge_best_pred)

print(f"Best Ridge alpha : {ridge_search.best_params_['model__alpha']}")
print(f"Tuned Ridge  MAE : {ridge_best_mae:,.0f}  |  R² : {ridge_best_r2:.4f}")
print(f"Default Ridge MAE: {ridge_mae:,.0f}  |  R² : {ridge_r2:.4f}")

In [ ]:
# ── Lasso tuning ──────────────────────────────────────────────────────────────
lasso_search = GridSearchCV(
    estimator  = Pipeline([('preprocessor', preprocessor),
                           ('model', Lasso(max_iter=10000))]),
    param_grid = {'model__alpha': [0.001, 0.01, 0.1, 1, 10]},
    cv         = tscv,
    scoring    = 'neg_mean_absolute_error',
    n_jobs     = -1
)
lasso_search.fit(X_train, y_train)
lasso_best_pred = lasso_search.best_estimator_.predict(X_test)
lasso_best_mae  = mean_absolute_error(y_test, lasso_best_pred)
lasso_best_r2   = r2_score(y_test, lasso_best_pred)

print(f"Best Lasso alpha : {lasso_search.best_params_['model__alpha']}")
print(f"Tuned Lasso  MAE : {lasso_best_mae:,.0f}  |  R² : {lasso_best_r2:.4f}")
print(f"Default Lasso MAE: {lasso_mae:,.0f}  |  R² : {lasso_r2:.4f}")

ridge_best_rmse = np.sqrt(mean_squared_error(y_test, ridge_best_pred))
ridge_best_mape = mean_absolute_percentage_error(y_test, ridge_best_pred)
lasso_best_rmse = np.sqrt(mean_squared_error(y_test, lasso_best_pred))
lasso_best_mape = mean_absolute_percentage_error(y_test, lasso_best_pred)

## 7 · Time-Series Forecasting (SARIMA) <a id='sarima'></a>

SARIMA is the **valid forecasting model** for this dataset.  
Unlike regression (which operates on individual rows), SARIMA works on  
**monthly aggregated totals** where temporal patterns can be captured.

Steps:
1. Aggregate to monthly totals  
2. Check stationarity (ADF test)  
3. Fit SARIMA(1,0,1)(1,0,1,12)  
4. Evaluate on last 12 months  
5. Forecast 2026

In [ ]:
# ── Aggregate to monthly level ────────────────────────────────────────────────
monthly_df = (
    df.groupby(['Year', 'Month'])['Estimated_Deliveries']
    .sum()
    .reset_index()
)
monthly_df['Date'] = pd.to_datetime(
    monthly_df['Year'].astype(str) + '-' + monthly_df['Month'].astype(str) + '-01'
)
monthly_df = monthly_df.sort_values('Date').set_index('Date')

print(f"Monthly series  : {len(monthly_df)} months ({monthly_df.index.min().date()} → {monthly_df.index.max().date()})")
print(f"Mean deliveries : {monthly_df['Estimated_Deliveries'].mean():,.0f}")
print(f"Std  deliveries : {monthly_df['Estimated_Deliveries'].std():,.0f}")

# ── Plot monthly trend ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_df.index, monthly_df['Estimated_Deliveries'], linewidth=2)
ax.set_title('Monthly Total Deliveries Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Monthly Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── ADF Stationarity Test ─────────────────────────────────────────────────────
adf_result = adfuller(monthly_df['Estimated_Deliveries'])

print("Augmented Dickey-Fuller Test")
print(f"  ADF Statistic : {adf_result[0]:.4f}")
print(f"  P-Value       : {adf_result[1]:.4f}")
print(f"  Critical 5%   : {adf_result[4]['5%']:.4f}")

if adf_result[1] < 0.05:
    print("\n✅ Series is stationary — d=0, no differencing needed.")
else:
    print("\n⚠️ Series is non-stationary — differencing required.")

In [ ]:
# ── Train/test split — hold out last 12 months ────────────────────────────────
train = monthly_df.iloc[:-12]
test  = monthly_df.iloc[-12:]

print(f"Training : {len(train)} months  ({train.index.min().date()} → {train.index.max().date()})")
print(f"Testing  : {len(test)} months  ({test.index.min().date()} → {test.index.max().date()})")

In [ ]:
# ── Fit SARIMA(1,0,1)(1,0,1,12) ──────────────────────────────────────────────
sarima_model = SARIMAX(
    train['Estimated_Deliveries'],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_results = sarima_model.fit(disp=False)

print(f"AIC : {sarima_results.aic:.2f}")
print(f"BIC : {sarima_results.bic:.2f}")
print("\n✅ SARIMA model fitted successfully.")

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
forecast_obj    = sarima_results.get_forecast(steps=12)
forecast_values = forecast_obj.predicted_mean
forecast_ci     = forecast_obj.conf_int()

sarima_mae  = mean_absolute_error(test['Estimated_Deliveries'], forecast_values)
sarima_rmse = np.sqrt(mean_squared_error(test['Estimated_Deliveries'], forecast_values))

print(f"SARIMA MAE  : {sarima_mae:,.0f}")
print(f"SARIMA RMSE : {sarima_rmse:,.0f}")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index[-24:], train['Estimated_Deliveries'].iloc[-24:],
        label='Training (last 24 months)', linewidth=2)
ax.plot(test.index, test['Estimated_Deliveries'],
        marker='o', label='Actual (test)', linewidth=2)
ax.plot(forecast_values.index, forecast_values,
        marker='o', linestyle='--', label='SARIMA Forecast', linewidth=2)
ax.fill_between(forecast_ci.index,
                forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                alpha=0.2, label='95% Confidence Interval')

ax.set_title('SARIMA — Forecast vs Actual (Test Period)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Monthly Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Forecast 2026 ─────────────────────────────────────────────────────────────
future_forecast   = sarima_results.get_forecast(steps=12)
future_pred       = future_forecast.predicted_mean
future_ci         = future_forecast.conf_int()
future_dates      = pd.date_range(
    start=monthly_df.index[-1] + pd.DateOffset(months=1),
    periods=12, freq='MS'
)

future_df = pd.DataFrame({
    'Forecasted_Deliveries' : future_pred.values,
    'Lower_95_CI'           : future_ci.iloc[:, 0].values,
    'Upper_95_CI'           : future_ci.iloc[:, 1].values,
}, index=future_dates)

display(future_df.round(0))

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(monthly_df.index[-36:], monthly_df['Estimated_Deliveries'].iloc[-36:],
        label='Historical (last 36 months)', linewidth=2)
ax.plot(future_df.index, future_df['Forecasted_Deliveries'],
        marker='o', linewidth=2, label='2026 Forecast')
ax.fill_between(future_df.index,
                future_df['Lower_95_CI'], future_df['Upper_95_CI'],
                alpha=0.2, label='95% Confidence Interval')

ax.set_title('Monthly Delivery Forecast — 2026', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Monthly Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Avg monthly forecast : {future_df['Forecasted_Deliveries'].mean():,.0f}")
print(f"Total 2026 forecast  : {future_df['Forecasted_Deliveries'].sum():,.0f}")

## 8 · Summary & Key Insights <a id='summary'></a>

| Stage | What we did | Result |
|-------|-------------|--------|
| **EDA** | Trends, model/region share, correlation heatmap | Leaky columns identified |
| **Preprocessing** | Dropped leakage, cyclical encoding, OHE, scaling | Clean pipeline |
| **Regression** | 4 models + baseline | Negative R² (expected — synthetic data) |
| **Hyperparameter Tuning** | Grid search + TimeSeriesSplit CV | Minor improvement |
| **SARIMA** | Monthly aggregation + seasonal model | Valid forecast for 2026 |

### Key Learning
> **Feature correlation is the foundation.**  
> If no feature correlates with the target, no model — no matter how complex — can predict well.  
> This is the most important lesson this notebook demonstrates.

### Why Regression Failed
After removing data leakage, all remaining features had correlation ≤ 0.03 with `Estimated_Deliveries`.  
This is a characteristic of the **synthetic dataset**, not a mistake in the pipeline.

### Why SARIMA Works
SARIMA operates directly on the time series of monthly totals.  
It learns temporal patterns (trend + seasonality) without needing external features.

In [ ]:
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)
print(f"\nDataset         : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Training period : {df.iloc[:split_index]['Year'].min()} – {df.iloc[:split_index]['Year'].max()}")
print(f"Test period     : {df.iloc[split_index:]['Year'].min()} – {df.iloc[split_index:]['Year'].max()}")
print(f"\nRegression Results (test set):")
print(f"  Best model R² : {max(lr_r2, ridge_r2, lasso_r2, rf_r2):.4f}  (near-zero due to synthetic data)")
print(f"  Best model MAE: {min(lr_mae, ridge_mae, lasso_mae, rf_mae):,.0f}")
print(f"\nSARIMA Results (test set):")
print(f"  MAE           : {sarima_mae:,.0f}")
print(f"  RMSE          : {sarima_rmse:,.0f}")
print(f"\n2026 Forecast   : {future_df['Forecasted_Deliveries'].sum():,.0f} total deliveries")

In [ ]:
# Save 2026 forecast
future_df.round(0).to_csv('delivery_forecast_2026.csv')
print("✅ Forecast saved to delivery_forecast_2026.csv")